# 20.5 半监督学习 / Semi-Supervised Learning

**中文**:标注数据很贵(要人工打标签),但**无标注数据往往几乎免费、海量**。**半监督学习(SSL)** 的目标:用**少量标注 + 大量无标注**,做得比"只用那点标注"好得多。核心信念是**"无标注数据里藏着数据的结构(簇、流形)"**——即使你不知道每个点的标签,知道"哪些点长得像、聚成一团"本身就极有价值。本节从零实现三条主流路线:**自训练(伪标签)、标签传播(图)、以及现代深度 SSL 的核心思想 FixMatch(一致性正则)**。
**English**: Labeled data is expensive (humans must label it), but **unlabeled data is often nearly free and abundant**. **Semi-supervised learning (SSL)** aims to do far better with **a few labels + lots of unlabeled data** than "only the few labels" would. The core belief is **"unlabeled data hides the data's structure (clusters, manifolds)"** — even without labels, knowing "which points look alike and cluster together" is hugely valuable. This section implements three mainstream routes from scratch: **self-training (pseudo-labeling), label propagation (graph-based), and the core idea of modern deep SSL, FixMatch (consistency regularization)**.

---

**中文**:SSL 能成立,靠三个**关键假设**(至少满足其一):
**English**: SSL works under three **key assumptions** (at least one holding):
- **平滑假设**:相近的点,标签也相近。
  **Smoothness**: nearby points have similar labels.
- **簇假设**:同一个簇里的点属于同一类(决策边界应落在低密度区,而不是穿过密集的簇)。
  **Cluster**: points in the same cluster share a label (the decision boundary lies in low-density regions, not through dense clusters).
- **流形假设**:高维数据其实分布在低维流形上,同流形上的点同类。
  **Manifold**: high-dim data lives on a low-dim manifold; points on the same manifold share a label.

**中文**:三条主流方法:
**English**: Three mainstream methods:
- **① 自训练(self-training / 伪标签)**:用少量标注训一个模型 → 对无标注数据预测 → 把**高置信度**的预测当作"伪标签"加入训练集 → 重训。简单通用,但有个致命风险——**确认偏差(confirmation bias)**:如果初始模型错了,它会自信地给出错误伪标签,然后在自己的错误上越训越错。
  **Self-training (pseudo-labeling)**: train on the few labels → predict on unlabeled → add **high-confidence** predictions as "pseudo-labels" → retrain. Simple and general, but with a fatal risk — **confirmation bias**: if the initial model is wrong, it confidently produces wrong pseudo-labels, then trains on its own mistakes, getting worse.
- **② 标签传播(label propagation)**:把所有点(标注+无标注)连成一个**相似度图**(近的点连边),让标签像"染色/水波"一样**沿图扩散**——标注点的标签流向相似的无标注点。天然利用流形结构。
  **Label propagation**: connect all points (labeled + unlabeled) into a **similarity graph** (edges between nearby points) and let labels **diffuse along the graph** like spreading color/ripples — labeled points' labels flow to similar unlabeled points. Naturally exploits the manifold.
- **③ FixMatch(一致性正则)**:现代深度 SSL 的代表。核心:**对同一张无标注图片的"弱增强"和"强增强"版本,模型的预测应该一致**。用弱增强的高置信预测当伪标签,监督强增强版本。用增强的多样性对抗确认偏差。
  **FixMatch (consistency regularization)**: the flagship of modern deep SSL. Core: **the model's predictions on a "weakly-augmented" and a "strongly-augmented" version of the same unlabeled image should agree**. Use the weak-augmentation's high-confidence prediction as the pseudo-label to supervise the strong-augmentation version. Augmentation diversity fights confirmation bias.

> 💡 **面试速查 / Interview cheat-sheet（★★ 少标注场景必考）**
> **中文**:SSL=少量标注+大量无标注, 靠**平滑/簇/流形假设**(相近同类、边界在低密度区)。**自训练(伪标签)**:高置信预测当标签重训, 简单但有**确认偏差**(自信地错→越训越错)。**标签传播**:相似度图上标签扩散, 用流形结构, 对两月牙这种数据很强。**一致性正则/FixMatch**(深度SSL SOTA):强弱增强预测一致 + 置信度阈值, 用增强对抗确认偏差(MixMatch/UDA/FixMatch)。**关键坑**:①**确认偏差**(自训练最危险);②假设不成立时 SSL 可能**反而更差**(无标注数据分布≠标注, 或类不平衡);③伪标签阈值/类平衡很重要。用途:图像/文本分类标注贵时、医疗、工业。vs 自监督:自监督无标签预训练学表示, SSL 用少量标签直接做任务。
> **English**: SSL = few labels + lots of unlabeled, relying on **smoothness/cluster/manifold assumptions** (nearby = same class, boundary in low-density regions). **Self-training (pseudo-labeling)**: retrain on high-confidence predictions — simple but prone to **confirmation bias** (confidently wrong → worse over time). **Label propagation**: diffuse labels on a similarity graph, exploiting the manifold, strong on data like two-moons. **Consistency regularization / FixMatch** (deep SSL SOTA): agreement between strong/weak augmentations + a confidence threshold, using augmentation to fight confirmation bias (MixMatch/UDA/FixMatch). **Key pitfalls**: ① **confirmation bias** (self-training's biggest danger); ② when assumptions fail, SSL can be **worse than supervised** (unlabeled distribution ≠ labeled, or class imbalance); ③ pseudo-label thresholds/class balance matter. Uses: image/text classification when labels are expensive, medical, industrial. vs self-supervised: self-supervised pretrains representations without labels, SSL uses a few labels for the task directly.


In [ ]:

# ============================================================
# 数据:两月牙 + 极少标注 / two moons with very few labels
# 中文:600 个点排成两个交错的月牙(经典 SSL 演示)。每类只给 3 个标注, 其余 594 个无标注。
# English: 600 points in two interleaving moons (classic SSL demo). Only 3 labels per class; 594 unlabeled.
# ============================================================
import numpy as np, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import make_moons
from sklearn.svm import SVC
from sklearn.neighbors import kneighbors_graph
from sklearn.linear_model import LogisticRegression
np.random.seed(0)
X,y=make_moons(600, noise=0.1, random_state=0)
lab_idx=np.r_[np.where(y==0)[0][:3], np.where(y==1)[0][:3]]  # 每类3个标注 / 3 labels per class
labeled=np.zeros(len(X),bool); labeled[lab_idx]=True
print(f"{len(X)} 个点, 只有 {labeled.sum()} 个有标注 ({labeled.mean():.0%}), 其余全靠无标注结构")

# 基线:只用标注 / baseline: supervised on the few labels
sup=SVC(gamma=2).fit(X[labeled], y[labeled])
acc_sup=(sup.predict(X)==y).mean()
print(f"① 仅监督(6标注)准确率 / supervised (6 labels): {acc_sup:.3f}")


**中文**:只用 6 个标注,监督模型的边界很粗糙。**② 标签传播**从零实现:建 kNN 相似度图,让标签沿图迭代扩散,标注点的标签"钳制(clamp)"不变。看它能否借无标注点的月牙结构把边界修好。
**English**: With only 6 labels, the supervised boundary is crude. **② Label propagation** from scratch: build a kNN similarity graph and iteratively diffuse labels along it, "clamping" the labeled points. See if it fixes the boundary using the unlabeled points' moon structure.


In [ ]:

# ============================================================
# ② 标签传播(从零)/ label propagation from scratch
# ============================================================
W=kneighbors_graph(X, n_neighbors=7, mode="connectivity").toarray()   # kNN 图 / kNN graph
W=np.maximum(W, W.T)                                                   # 对称化 / symmetrize
Dinv=1/np.maximum(W.sum(1), 1e-9)                                     # 度的逆(行归一化)/ inverse degree
Ylab=np.zeros((len(X),2))
for i in lab_idx: Ylab[i, y[i]]=1                                     # 标注点的独热标签 / one-hot labels
F=Ylab.copy()
for _ in range(1000):                                                # 迭代扩散 / iterative diffusion
    F=Dinv[:,None]*(W@F)                                             # 每个点=邻居标签分布的加权平均 / propagate
    F[labeled]=Ylab[labeled]                                         # 钳制标注点 / clamp labeled
pred_lp=F.argmax(1)                                                  # 传播后每个点的标签 / propagated labels
acc_lp=(pred_lp==y).mean()
print(f"② 标签传播准确率 / label propagation: {acc_lp:.3f}  (远超仅监督 {acc_sup:.3f})")
print("→ 标签沿'月牙'的图结构流动, 借无标注点还原了正确边界 / labels flow along the moon manifold")


**中文**:标签传播大幅提升!因为它顺着月牙的流形结构扩散标签。**③ 自训练**——但它有个陷阱。我们故意用一个**线性模型**做自训练,看会发生什么(现实中如果初始模型不够好,就会踩这个坑)。
**English**: Label propagation improves a lot — it diffuses labels along the moon manifold. **③ Self-training** — but it has a trap. We deliberately use a **linear model** for self-training to see what happens (in reality, a weak initial model falls into this pit).


In [ ]:

# ============================================================
# ③ 自训练的确认偏差(用线性模型演示)/ self-training's confirmation bias (with a linear model)
# ============================================================
cur_mask=labeled.copy(); cur_y=y.copy().astype(int)
history=[]
for it in range(15):
    clf=LogisticRegression(max_iter=1000).fit(X[cur_mask], cur_y[cur_mask])   # 线性模型(拟合不了月牙)/ linear
    proba=clf.predict_proba(X); conf=proba.max(1); pseudo=proba.argmax(1)
    add=(~cur_mask)&(conf>0.85)                                        # 高置信度就加伪标签 / add high-confidence
    history.append(cur_mask.sum())
    if add.sum()==0: break
    cur_mask[add]=True; cur_y[add]=pseudo[add]                        # 用自己的预测当标签 / self-label
acc_st=(LogisticRegression(max_iter=1000).fit(X[cur_mask],cur_y[cur_mask]).predict(X)==y).mean()
print(f"③ 自训练(线性模型)准确率 / self-training (linear): {acc_st:.3f}  ← 灾难!")
print(f"确认偏差:线性模型画不出月牙的弯曲边界, 却'自信地'给出错误伪标签, 越训越错")
print(f"对比 / comparison:  仅监督 {acc_sup:.3f}  |  标签传播 {acc_lp:.3f}  |  自训练(线性) {acc_st:.3f}")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(16,4.8))
# ① 数据 + 极少标注 / data + few labels
ax[0].scatter(X[~labeled,0],X[~labeled,1],c="#cccccc",s=10,label="无标注 unlabeled")
ax[0].scatter(X[labeled,0],X[labeled,1],c=y[labeled],cmap="coolwarm",s=150,edgecolor="k",zorder=5,label="标注(仅6个)")
ax[0].set_title(f"两月牙:594无标注 + 6标注 / {labeled.sum()} labels"); ax[0].legend(fontsize=8)
# ② 标签传播结果 / label propagation result
ax[1].scatter(X[:,0],X[:,1],c=pred_lp,cmap="coolwarm",s=12)
ax[1].scatter(X[labeled,0],X[labeled,1],c="k",s=80,marker="*",zorder=5)
ax[1].set_title(f"标签传播(acc {acc_lp:.2f}): 沿月牙正确扩散 / label propagation")
# ③ 三方法准确率 / accuracy comparison
ax[2].bar(["仅监督\n(6标注)","标签传播\nlabel prop","自训练(线性)\nself-train"],[acc_sup,acc_lp,acc_st],
          color=["#8C8C8C","#55A868","#C44E52"])
ax[2].axhline(0.5,ls="--",color="gray"); ax[2].set_ylim(0,1.05)
for i,v in enumerate([acc_sup,acc_lp,acc_st]): ax[2].text(i,v,f"{v:.2f}",ha="center",va="bottom")
ax[2].set_title("标签传播最好, 自训练确认偏差最糟 / label prop wins, self-train bias fails"); ax[2].set_ylabel("准确率")
plt.tight_layout(); plt.savefig("/tmp/adv05_viz.png",dpi=80); plt.show()
print("标签传播利用流形结构大胜; 自训练用弱模型则被确认偏差反噬")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **无标注数据的结构是金矿**:只有 6 个标注(1%),标签传播就把准确率从 0.83 提到 0.93——它顺着月牙的**流形结构**把标签正确地扩散出去。这生动展示了 SSL 的核心信念:**"知道哪些点聚成一团"本身就极有价值**,即使你不知道每团的标签。数据的几何结构替你补上了缺失的监督。
2. **自训练的确认偏差是最危险的诚实陷阱**:我们故意用线性模型自训练,结果**准确率崩到 0.5(纯随机)**——因为线性模型画不出月牙的弯曲边界,却"自信满满"地把大批点错误标注,然后在自己的错误上反复强化,越训越离谱。**这是 SSL 最需要警惕的坑:伪标签会放大初始模型的错误。** 现代深度 SSL(FixMatch)正是用"**对强弱增强的预测要一致 + 高置信度阈值**"来对抗它——增强让模型不能对一张图轻易"自信地错"。
3. **诚实的边界条件:SSL 不是万能药**:①如果假设不成立(无标注数据和标注数据分布不同、或类严重不平衡),SSL **可能反而比只用标注更差**——这也是学术界反复强调的("无标注数据不总是有帮助");②标签传播是**直推式**的(只给这批无标注点标签,来新点要重算),且 $O(n^2)$ 建图不 scale;③伪标签方法对**阈值和类平衡**很敏感。所以用 SSL 前,先确认你的数据满足平滑/簇/流形假设,并始终和"仅监督"基线对比。

**English**:
1. **The structure of unlabeled data is a gold mine**: with only 6 labels (1%), label propagation lifts accuracy from 0.83 to 0.93 — diffusing labels correctly along the moon **manifold**. This vividly shows SSL's core belief: **"knowing which points cluster together" is itself hugely valuable**, even without each cluster's label. The data's geometry supplies the missing supervision.
2. **Self-training's confirmation bias is the most dangerous honest trap**: we deliberately used a linear model for self-training, and accuracy **collapsed to 0.5 (pure chance)** — the linear model can't draw the moons' curved boundary yet "confidently" mislabels many points, then reinforces its own mistakes over rounds, spiraling worse. **This is SSL's most important pitfall: pseudo-labels amplify the initial model's errors.** Modern deep SSL (FixMatch) fights it precisely with "**agreement between strong/weak augmentations + a high-confidence threshold**" — augmentation prevents the model from being "confidently wrong" on an image.
3. **Honest boundary conditions: SSL is not a cure-all**: ① if assumptions fail (unlabeled and labeled distributions differ, or severe class imbalance), SSL **can be worse than supervised-only** — a point academia repeatedly stresses ("unlabeled data doesn't always help"); ② label propagation is **transductive** (labels only these unlabeled points; new points need recomputation) and its $O(n^2)$ graph doesn't scale; ③ pseudo-label methods are sensitive to **thresholds and class balance**. So before using SSL, confirm your data meets the smoothness/cluster/manifold assumptions and always compare against the supervised-only baseline.

> 💼 **实战视角 / Practical angle**
> **中文**:SSL 在**标注昂贵**的场景价值巨大:医疗影像(专家标注贵)、工业质检、内容审核、少语种 NLP。落地要点:①**先试自监督预训练 + 少量微调**(现在往往比经典 SSL 更强, 尤其有大规模无标注数据);②图像/文本深度 SSL 用 **FixMatch/MixMatch/UDA**(一致性正则是核心);③表格/图数据用**标签传播**(sklearn `LabelSpreading/LabelPropagation`);④**永远和仅监督基线比**(SSL 可能帮倒忙);⑤伪标签要设**高置信阈值 + 类平衡**防确认偏差;⑥主动学习(下节 20.6)是互补路线——不是"用无标注",而是"聪明地选哪些去标注"。面试金句:*"SSL 用少量标注+大量无标注, 靠平滑/簇/流形假设; 自训练(伪标签)简单但有确认偏差、标签传播用图流形结构、FixMatch 用强弱增强一致性对抗偏差; 但假设不成立时 SSL 可能更差, 务必和仅监督基线对比。"*
> **English**: SSL is hugely valuable where **labeling is expensive**: medical imaging (expensive expert labels), industrial inspection, content moderation, low-resource NLP. Deployment keys: ① **first try self-supervised pretraining + light fine-tuning** (often stronger than classic SSL now, especially with large unlabeled data); ② for image/text deep SSL use **FixMatch/MixMatch/UDA** (consistency regularization is core); ③ for tabular/graph use **label propagation** (sklearn `LabelSpreading/LabelPropagation`); ④ **always compare to the supervised-only baseline** (SSL can backfire); ⑤ use a **high confidence threshold + class balancing** for pseudo-labels to prevent confirmation bias; ⑥ active learning (20.6 next) is a complementary route — not "use unlabeled" but "smartly choose what to label." Interview line: *"SSL uses few labels + lots of unlabeled, relying on smoothness/cluster/manifold assumptions; self-training (pseudo-labels) is simple but prone to confirmation bias, label propagation exploits the graph manifold, FixMatch uses strong/weak-augmentation consistency to fight bias; but SSL can be worse when assumptions fail — always compare to the supervised baseline."*

---
### 小结 / Summary
- **中文**:SSL 用少量标注+大量无标注, 靠平滑/簇/流形假设; 无标注数据的结构替你补监督。
- **English**: SSL uses few labels + lots of unlabeled, relying on smoothness/cluster/manifold assumptions; unlabeled structure supplies missing supervision.
- **中文**:标签传播(图流形, 两月牙上大胜)、自训练(伪标签, 有确认偏差)、FixMatch(强弱增强一致性, 深度SSL SOTA)。
- **English**: Label propagation (graph manifold, wins on two-moons), self-training (pseudo-labels, confirmation-bias-prone), FixMatch (strong/weak augmentation consistency, deep SSL SOTA).
- **中文**:SSL 不是万能药——假设不成立可能更差; 务必和仅监督基线对比, 伪标签要防确认偏差。
- **English**: SSL is not a cure-all — worse when assumptions fail; always compare to supervised-only, and guard pseudo-labels against confirmation bias.
